# Sentiment Analyzer with HuggingFace

Analyze the sentiment of texts and reviews using HuggingFace Transformers pipelines.
Supports both general sentiment and financial sentiment analysis.

**What this project demonstrates:**
- HuggingFace Transformers `pipeline` API for quick inference
- AutoTokenizer for understanding tokenization
- Using pre-trained models (no training required)
- Comparing different sentiment models on the same input
- Gradio interface for interactive analysis

Built as part of Module 5 (HuggingFace 1) of the LLM & Agentic AI Bootcamp.

**Note:** This notebook runs on CPU. For GPU-heavy models, use Google Colab.

## Setup

In [1]:
# Install if needed (uncomment)
# !pip install transformers torch gradio

from transformers import pipeline, AutoTokenizer
import gradio as gr
from IPython.display import display, Markdown

print("Libraries loaded!")

/Users/eugenionex/Downloads/LLM and Agentic AI Bootcamp Materials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded!


## Understanding Tokenization

Before sentiment analysis, let's see how models "read" text.
Tokenizers split text into tokens (subwords) that the model understands.

In [2]:
# Load a tokenizer to see how text gets broken down
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

sample = "HuggingFace makes NLP incredibly easy!"
tokens = tokenizer.tokenize(sample)
token_ids = tokenizer.encode(sample)

print(f"Original text: {sample}")
print(f"Tokens:        {tokens}")
print(f"Token IDs:     {token_ids}")
print(f"Num tokens:    {len(tokens)}")

Original text: HuggingFace makes NLP incredibly easy!
Tokens:        ['hugging', '##face', 'makes', 'nl', '##p', 'incredibly', 'easy', '!']
Token IDs:     [101, 17662, 12172, 3084, 17953, 2361, 11757, 3733, 999, 102]
Num tokens:    8


## Loading Sentiment Models

HuggingFace `pipeline` makes it trivial to load pre-trained models.
We'll use two different models:
- **General sentiment** (positive/negative) for product reviews, tweets, etc.
- **Financial sentiment** (positive/negative/neutral) for financial texts

In [3]:
# General sentiment model (fast, runs on CPU)
general_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("General sentiment model loaded!")

# Financial sentiment model (FinBERT by ProsusAI)
financial_analyzer = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert"
)
print("Financial sentiment model loaded!")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7895.02it/s]


General sentiment model loaded!


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26177.77it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Financial sentiment model loaded!


## Quick Test

Let's test both models on different types of text.

In [4]:
# General sentiment - product reviews
reviews = [
    "This product is absolutely amazing, best purchase I've ever made!",
    "Terrible quality. Broke after one day. Total waste of money.",
    "It's okay, nothing special but it works."
]

print("=== General Sentiment Analysis ===")
for review in reviews:
    result = general_analyzer(review)[0]
    print(f"\n\"{review[:60]}...\"")
    print(f"  → {result['label']} (confidence: {result['score']:.2%})")

=== General Sentiment Analysis ===

"This product is absolutely amazing, best purchase I've ever ..."
  → POSITIVE (confidence: 99.99%)

"Terrible quality. Broke after one day. Total waste of money...."
  → NEGATIVE (confidence: 99.98%)

"It's okay, nothing special but it works...."
  → POSITIVE (confidence: 99.98%)


In [5]:
# Financial sentiment - earnings & market news
financial_texts = [
    "Revenue grew 15% year-over-year, beating analyst expectations.",
    "The company reported a significant loss and is considering layoffs.",
    "Q3 results were in line with market expectations, no surprises."
]

print("=== Financial Sentiment Analysis (FinBERT) ===")
for text in financial_texts:
    result = financial_analyzer(text)[0]
    print(f"\n\"{text[:60]}...\"")
    print(f"  → {result['label']} (confidence: {result['score']:.2%})")

=== Financial Sentiment Analysis (FinBERT) ===

"Revenue grew 15% year-over-year, beating analyst expectation..."
  → positive (confidence: 95.81%)

"The company reported a significant loss and is considering l..."
  → negative (confidence: 96.66%)

"Q3 results were in line with market expectations, no surpris..."
  → positive (confidence: 89.55%)


## Batch Analysis

Analyze multiple texts at once and get a summary.

In [6]:
def analyze_batch(texts, analyzer, analyzer_name="Model"):
    """Analyze a list of texts and return a summary."""
    results = analyzer(texts)
    
    lines = [f"## {analyzer_name} - Batch Analysis\n"]
    lines.append("| # | Text (preview) | Sentiment | Confidence |")
    lines.append("|---|----------------|-----------|------------|")
    
    label_counts = {}
    for i, (text, result) in enumerate(zip(texts, results), 1):
        label = result["label"]
        score = result["score"]
        preview = text[:50].replace("|", "/")
        lines.append(f"| {i} | {preview}... | **{label}** | {score:.2%} |")
        label_counts[label] = label_counts.get(label, 0) + 1
    
    lines.append("\n### Distribution")
    for label, count in sorted(label_counts.items()):
        pct = count / len(texts) * 100
        lines.append(f"- **{label}:** {count}/{len(texts)} ({pct:.0f}%)")
    
    display(Markdown("\n".join(lines)))
    return results

In [7]:
# Analyze a batch of app store reviews
app_reviews = [
    "Love this app! So intuitive and fast.",
    "Keeps crashing on my phone. Very frustrating.",
    "Great concept but needs more features.",
    "Best productivity app I've used in years.",
    "The latest update ruined everything.",
    "Does what it says. Simple and effective.",
    "Can't even log in anymore after the update.",
    "Customer support was incredibly helpful!"
]

results = analyze_batch(app_reviews, general_analyzer, "General Sentiment")

## General Sentiment - Batch Analysis

| # | Text (preview) | Sentiment | Confidence |
|---|----------------|-----------|------------|
| 1 | Love this app! So intuitive and fast.... | **POSITIVE** | 99.99% |
| 2 | Keeps crashing on my phone. Very frustrating.... | **NEGATIVE** | 99.83% |
| 3 | Great concept but needs more features.... | **NEGATIVE** | 95.60% |
| 4 | Best productivity app I've used in years.... | **POSITIVE** | 99.92% |
| 5 | The latest update ruined everything.... | **NEGATIVE** | 99.98% |
| 6 | Does what it says. Simple and effective.... | **POSITIVE** | 99.99% |
| 7 | Can't even log in anymore after the update.... | **NEGATIVE** | 99.92% |
| 8 | Customer support was incredibly helpful!... | **POSITIVE** | 99.90% |

### Distribution
- **NEGATIVE:** 4/8 (50%)
- **POSITIVE:** 4/8 (50%)

## Compare Models

Same text, two models - see how general vs financial models interpret differently.

In [8]:
def compare_models(text):
    """Compare general vs financial sentiment on the same text."""
    general = general_analyzer(text)[0]
    financial = financial_analyzer(text)[0]
    
    lines = [
        f"**Text:** \"{text}\"\n",
        "| Model | Sentiment | Confidence |",
        "|-------|-----------|------------|",
        f"| General (DistilBERT) | **{general['label']}** | {general['score']:.2%} |",
        f"| Financial (FinBERT) | **{financial['label']}** | {financial['score']:.2%} |"
    ]
    display(Markdown("\n".join(lines)))

In [9]:
# Interesting comparison: a text that could be read differently
compare_models("The stock dropped 5% after the CEO announced a major restructuring plan.")
print()
compare_models("Operating costs were reduced by 20% through strategic automation.")
print()
compare_models("The company is taking on more debt to fund expansion.")

**Text:** "The stock dropped 5% after the CEO announced a major restructuring plan."

| Model | Sentiment | Confidence |
|-------|-----------|------------|
| General (DistilBERT) | **NEGATIVE** | 99.93% |
| Financial (FinBERT) | **negative** | 96.96% |

**Text:** "Operating costs were reduced by 20% through strategic automation."

| Model | Sentiment | Confidence |
|-------|-----------|------------|
| General (DistilBERT) | **NEGATIVE** | 99.79% |
| Financial (FinBERT) | **negative** | 84.24% |

**Text:** "The company is taking on more debt to fund expansion."

| Model | Sentiment | Confidence |
|-------|-----------|------------|
| General (DistilBERT) | **NEGATIVE** | 99.45% |
| Financial (FinBERT) | **neutral** | 57.27% |

## Gradio Interface

Interactive UI where you can type any text and see both models' analysis.

In [10]:
def analyze_sentiment_ui(text, model_choice):
    """Analyze sentiment based on selected model."""
    if not text.strip():
        return "Please enter some text to analyze."
    
    if model_choice == "General (Reviews, Tweets)":
        result = general_analyzer(text)[0]
        model_name = "DistilBERT (General)"
    elif model_choice == "Financial (Earnings, Market)":
        result = financial_analyzer(text)[0]
        model_name = "FinBERT (Financial)"
    else:  # Both
        gen = general_analyzer(text)[0]
        fin = financial_analyzer(text)[0]
        return (
            f"## General Model (DistilBERT)\n"
            f"**{gen['label']}** — Confidence: {gen['score']:.2%}\n\n"
            f"## Financial Model (FinBERT)\n"
            f"**{fin['label']}** — Confidence: {fin['score']:.2%}"
        )
    
    return f"## {model_name}\n**{result['label']}** — Confidence: {result['score']:.2%}"


demo = gr.Interface(
    fn=analyze_sentiment_ui,
    inputs=[
        gr.Textbox(lines=4, placeholder="Enter text to analyze...", label="Text"),
        gr.Radio(
            choices=["General (Reviews, Tweets)", "Financial (Earnings, Market)", "Both"],
            value="Both",
            label="Model"
        )
    ],
    outputs=gr.Markdown(label="Sentiment Analysis"),
    title="Sentiment Analyzer",
    description="Analyze text sentiment using HuggingFace pre-trained models. Compare general vs financial sentiment.",
    flagging_mode="never",
    examples=[
        ["This product exceeded all my expectations. Highly recommended!", "General (Reviews, Tweets)"],
        ["Revenue increased 25% YoY driven by strong cloud adoption.", "Financial (Earnings, Market)"],
        ["The company announced layoffs while reporting record profits.", "Both"]
    ]
)

print("Launching Sentiment Analyzer...")
demo.launch()

Launching Sentiment Analyzer...
* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Key Takeaways

1. **HuggingFace `pipeline`** is the fastest way to use pre-trained models - one line of code
2. **AutoTokenizer** shows how models convert text to numbers (tokens)
3. **Domain-specific models** (like FinBERT) outperform general models on specialized text
4. **No training needed** - thousands of pre-trained models are ready to use on HuggingFace Hub
5. These models run **locally** - no API keys, no costs, full privacy